# Titanic Passenger Survival: Data Acquisition and Preparation

**Course:** ANA 500 Python for Data Science
**Assignment:** Micro-Project 1
**Name** Travis James
**Dataset:** Titanic passenger data
**Scope:** Steps 1 and 2 of the data science process (Acquire and Prepare) using NumPy and Pandas

---

## Problem Statement

Survival on the Titanic was not random. This project aims to describe which passenger characteristics (such as sex, ticket class, age, and family situation) were associated with who lived and who died, using the passenger manifest data.

## Hypothesis

Women and passengers in higher ticket classes survived at higher rates than men and passengers in lower ticket classes.

The hypothesis depends mainly on `Sex`, `Pclass`, and `Survived`, with `Age` and family size as supporting variables. The goal of this notebook is to get the data into a clean, trustworthy state so those variables can be analyzed in later steps.

---
# Step 1: Acquire

Identify the dataset, retrieve it, and take a first look at its structure.

**Source:** `titanic.csv`, one of the curated datasets provided for the course. Each row is one passenger, and each column is an attribute of that passenger.

In [2]:
# Import the two core libraries used in this micro-project.

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [3]:
# Load the CSV file into a Pandas DataFrame.
DATA_PATH = "titanic.csv"
raw = pd.read_csv(DATA_PATH)

print(f"Rows: {raw.shape[0]:,}   Columns: {raw.shape[1]}")

raw.head()

Rows: 1,309   Columns: 12


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Data dictionary

| Column | Meaning |
|---|---|
| `PassengerId` | Unique row identifier |
| `Survived` | Survival outcome (0 = died, 1 = survived) |
| `Pclass` | Ticket class (1 = first, 2 = second, 3 = third), a rough proxy for socio-economic status |
| `Name` | Passenger name, including title (Mr, Mrs, Miss, and so on) |
| `Sex` | male or female |
| `Age` | Age in years (fractional for infants) |
| `SibSp` | Number of siblings or spouses aboard |
| `Parch` | Number of parents or children aboard |
| `Ticket` | Ticket number |
| `Fare` | Ticket price paid |
| `Cabin` | Cabin number |
| `Embarked` | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |

In [4]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     891 non-null    float64
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    object 
 11  Embarked     1307 non-null   object 
dtypes: float64(3), int64(4), object(5)
memory usage: 122.8+ KB


In [5]:
n_labeled = raw["Survived"].notna().sum()
n_unlabeled = raw["Survived"].isna().sum()
print(f"Rows with a known outcome (Survived = 0 or 1): {n_labeled}")
print(f"Rows with a blank outcome                     : {n_unlabeled}")

# Find where the blank rows sit by looking at their PassengerId range.
blank_ids = raw.loc[raw["Survived"].isna(), "PassengerId"]
print(f"Blank outcomes span PassengerId {blank_ids.min()} to {blank_ids.max()}")

Rows with a known outcome (Survived = 0 or 1): 891
Rows with a blank outcome                     : 418
Blank outcomes span PassengerId 892 to 1309


**Acquire findings**

- The file contains the full passenger list (1,309 rows), not only passengers with a known outcome.
- 418 rows (PassengerId 892 to 1309) have no `Survived` value. These come from the holdout portion of the original dataset. They cannot be used to test a hypothesis about survival, so the analysis frame will later be limited to the 891 rows with a known outcome.
- All 1,309 rows are still worth cleaning, because the imputation logic for `Age` and `Fare` benefits from the full set of passengers and never uses the `Survived` column.

---
# Step 2: Prepare

Explore the data, find its quality problems, and pre-process it. This step has three parts:

1. **Explore:** summary statistics, missing values, duplicates, and suspicious values
2. **Clean and transform:** fix each problem with a documented decision
3. **Validate:** prove the cleaned data meets expectations, then save it

### 2a. Explore

In [6]:
raw.describe().T

,count,mean,std,min,25%,50%,75%,max
PassengerId,1309.0,655.000000,378.020061,1.00,328.0000,655.0000,982.000,1309.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.000,1.0000
Pclass,1309.0,2.294882,0.837836,1.00,2.0000,3.0000,3.000,3.0000
Age,1046.0,29.881138,14.413493,0.17,21.0000,28.0000,39.000,80.0000
SibSp,1309.0,0.498854,1.041658,0.00,0.0000,0.0000,1.000,8.0000
Parch,1309.0,0.385027,0.865560,0.00,0.0000,0.0000,0.000,9.0000
Fare,1308.0,33.295479,51.758668,0.00,7.8958,14.4542,31.275,512.3292


In [7]:
raw[["Name", "Sex", "Ticket", "Cabin", "Embarked"]].describe().T

,count,unique,top,freq
Name,1309,1307,"Connolly, Miss. Kate",2
Sex,1309,2,male,843
Ticket,1309,929,CA. 2343,11
Cabin,295,186,C23 C25 C27,6
Embarked,1307,3,S,914


In [8]:
for col in ["Pclass", "Sex", "Embarked"]:
    print(f"--- {col} ---")
    print(raw[col].value_counts(dropna=False))
    print()

--- Pclass ---
Pclass
3    709
1    323
2    277
Name: count, dtype: int64

--- Sex ---
Sex
male      843
female    466
Name: count, dtype: int64

--- Embarked ---
Embarked
S      914
C      270
Q      123
NaN      2
Name: count, dtype: int64



In [9]:

missing_count = raw.isna().sum()
missing_pct = (missing_count / len(raw) * 100).round(1)

missing_report = (
    pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
    .query("missing_count > 0")                       # keep only columns that actually have gaps
    .sort_values("missing_count", ascending=False)    # worst columns first
)
missing_report

,missing_count,missing_pct
Cabin,1014,77.5
Survived,418,31.9
Age,263,20.1
Embarked,2,0.2
Fare,1,0.1


**Missing value findings**

- `Cabin` is missing for about 77% of passengers. Filling that many blanks would mean inventing data, so the better move is to extract what is known (the deck letter) and flag the rest.
- `Age` is missing for about 20% of passengers, which is too many to drop and too important to ignore.
- `Embarked` (2 rows) and `Fare` (1 row) have only a handful of gaps and can be filled with defensible values.
- `Survived` is missing for the 418 holdout rows found in Step 1.

In [10]:
age_missing_rate = (
    raw.assign(age_missing=raw["Age"].isna())
       .groupby(["Pclass", "Sex"])["age_missing"]
       .mean()
       .mul(100)
       .round(1)
       .unstack()      
)
print("Percent of Age values missing, by ticket class and sex:")
age_missing_rate

Percent of Age values missing, by ticket class and sex:


Sex,female,male
Pclass,,
1,7.6,15.6
2,2.8,7.6
3,29.6,29.2


Age is missing far more often in third class than in first or second class. Because the gaps are not evenly spread, filling them with one overall median would pull third-class ages toward the wrong value. The imputation in Step 2b uses group medians instead.

In [11]:
# --- Duplicate checks ---

# 1) Fully identical rows.
print("Fully duplicated rows:", raw.duplicated().sum())

# 2) PassengerId should be unique because it is the row identifier.
print("PassengerId is unique:", raw["PassengerId"].is_unique)

# 3) Repeated names. A repeated name could be a data-entry duplicate or two different people.
#    keep=False marks every copy so both rows are displayed for comparison.
repeated_names = raw[raw["Name"].duplicated(keep=False)].sort_values("Name")
repeated_names[["PassengerId", "Survived", "Name", "Age", "Ticket", "Fare"]]

Fully duplicated rows: 0
PassengerId is unique: True


,PassengerId,Survived,Name,Age,Ticket,Fare
289,290,1.0,"Connolly, Miss. Kate",22.0,370373,7.7500
897,898,NaN,"Connolly, Miss. Kate",30.0,330972,7.6292
696,697,0.0,"Kelly, Mr. James",44.0,363592,8.0500
891,892,NaN,"Kelly, Mr. James",34.5,330911,7.8292


Two names appear twice, but each pair has a different ticket, age, and fare. They are different passengers who happen to share a name, so no rows are removed.

In [14]:
# --- Validity and suspicious-value checks ---

# Category columns should only contain expected values.
print("Pclass only contains 1, 2, 3   :", np.isin(raw["Pclass"], [1, 2, 3]).all())
print("Sex only contains male/female  :", np.isin(raw["Sex"], ["male", "female"]).all())
print("Embarked only contains C, Q, S :", np.isin(raw["Embarked"].dropna(), ["C", "Q", "S"]).all())


print("SibSp and Parch are non-negative:", bool((raw[["SibSp", "Parch"]] >= 0).all().all()))


print()
print("Passengers with Fare = 0:", (raw["Fare"] == 0).sum())
print("Passengers with Age < 1 :", (raw["Age"] < 1).sum())


q1, q3 = np.percentile(raw["Fare"].dropna(), [25, 75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
print()
print(f"Fare Q1 = {q1:.2f}, Q3 = {q3:.2f}, upper fence = {upper_fence:.2f}")
print("Fares above the upper fence:", (raw["Fare"] > upper_fence).sum())
print(f"Fare skewness: {raw['Fare'].skew():.2f}")

Pclass only contains 1, 2, 3   : True
Sex only contains male/female  : True
Embarked only contains C, Q, S : True
SibSp and Parch are non-negative: True

Passengers with Fare = 0: 17
Passengers with Age < 1 : 12

Fare Q1 = 7.90, Q3 = 31.27, upper fence = 66.34
Fares above the upper fence: 171
Fare skewness: 4.37


**Validity findings**

- All categorical columns contain only expected values, and no counts are negative.
- 17 passengers have a fare of 0. These are plausibly unrecorded fares or complimentary tickets, so they are kept and flagged rather than altered.
- The passengers under 1 year old are real infants with fractional ages, so they are valid.
- `Fare` is strongly right-skewed, with 171 fares above the IQR upper fence stretching a long right tail. These are real ticket prices, so they stay in the data, but a log transform will make the column better behaved for later steps.

### 2b. Clean and transform

In [15]:
clean = raw.copy()

clean["Survived"] = clean["Survived"].astype("Int64")
clean["Survived"].value_counts(dropna=False)

Survived
0       549
<NA>    418
1       342
Name: count, dtype: Int64

In [16]:
print(clean.loc[clean["Embarked"].isna(), ["PassengerId", "Pclass", "Ticket", "Fare", "Cabin"]])


embarked_mode = clean["Embarked"].mode()[0]
print("\nMost common port:", embarked_mode)

clean["Embarked"] = clean["Embarked"].fillna(embarked_mode)
print("Embarked missing after fix:", clean["Embarked"].isna().sum())

     PassengerId  Pclass  Ticket  Fare Cabin
61            62       1  113572  80.0   B28
829          830       1  113572  80.0   B28

Most common port: S
Embarked missing after fix: 0


In [17]:
clean["FareWasZero"] = clean["Fare"].eq(0)


median_fare_by_class = clean.groupby("Pclass")["Fare"].transform("median")
clean["Fare"] = clean["Fare"].fillna(median_fare_by_class)
print("Fare missing after fix:", clean["Fare"].isna().sum())

Fare missing after fix: 0


In [18]:

clean["Title"] = clean["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
print("Raw titles found:")
print(clean["Title"].value_counts())

Raw titles found:
Title
Mr              757
Miss            260
Mrs             197
Master           61
Rev               8
Dr                8
Col               4
Mlle              2
Major             2
Ms                2
Lady              1
Sir               1
Mme               1
Don               1
Capt              1
the Countess      1
Jonkheer          1
Dona              1
Name: count, dtype: int64


In [19]:

clean["Title"] = clean["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})


common_titles = ["Mr", "Miss", "Mrs", "Master"]
clean["Title"] = np.where(clean["Title"].isin(common_titles), clean["Title"], "Rare")

clean["Title"].value_counts()

Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64

In [20]:

clean["AgeWasMissing"] = clean["Age"].isna()


print("Median age by class and title (from passengers with a known age):")
print(clean.groupby(["Pclass", "Title"])["Age"].median().unstack().round(1))

Median age by class and title (from passengers with a known age):
Title   Master  Miss    Mr   Mrs  Rare
Pclass                                
1          6.0  30.0  41.5  45.0  48.5
2          2.0  20.0  30.0  30.5  41.5
3          6.0  18.0  26.0  31.0   NaN


In [21]:


# Level 1: median of the same Pclass and Title.
median_by_class_title = clean.groupby(["Pclass", "Title"])["Age"].transform("median")
clean["Age"] = clean["Age"].fillna(median_by_class_title)

# Level 2: median of the same Pclass, in case a class and title group had no known ages at all.
clean["Age"] = clean["Age"].fillna(clean.groupby("Pclass")["Age"].transform("median"))

# Level 3: overall median, as a final safety net.
clean["Age"] = clean["Age"].fillna(clean["Age"].median())

print("Age missing after fix:", clean["Age"].isna().sum())
print("Ages that were imputed:", clean["AgeWasMissing"].sum())

Age missing after fix: 0
Ages that were imputed: 263


In [22]:
# Check that imputing did not distort the age distribution.
age_check = pd.DataFrame({
    "before_imputation": raw["Age"].describe(),
    "after_imputation": clean["Age"].describe(),
}).round(2)
age_check

,before_imputation,after_imputation
count,1046.00,1309.00
mean,29.88,29.27
std,14.41,13.45
min,0.17,0.17
25%,21.00,21.00
50%,28.00,26.00
75%,39.00,36.50
max,80.00,80.00


In [23]:
# Too much is missing to fill in cabin numbers, so two simpler signals are extracted:
#   HasCabin : 1 if a cabin was recorded, 0 if not (a recorded cabin often means a first-class passenger)
#   Deck     : the first letter of the cabin (A to G, T), with "U" for Unknown
clean["HasCabin"] = np.where(clean["Cabin"].notna(), 1, 0)
clean["Deck"] = clean["Cabin"].str[0].fillna("U")

# The raw Cabin column is no longer needed once its information has been extracted.
clean = clean.drop(columns=["Cabin"])

clean["Deck"].value_counts()

Deck
U    1014
C      94
B      65
D      46
E      41
A      22
F      21
G       5
T       1
Name: count, dtype: int64

In [24]:
clean["FamilySize"] = clean["SibSp"] + clean["Parch"] + 1

clean["IsAlone"] = np.where(clean["FamilySize"] == 1, 1, 0)

clean[["SibSp", "Parch", "FamilySize", "IsAlone"]].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


In [25]:

age_conditions = [
    clean["Age"] < 13,      # Child
    clean["Age"] < 18,      # Teen
    clean["Age"] < 40,      # Adult
    clean["Age"] < 60,      # MiddleAge
]
age_labels = ["Child", "Teen", "Adult", "MiddleAge"]
clean["AgeGroup"] = np.select(age_conditions, age_labels, default="Senior")

clean["AgeGroup"].value_counts()

AgeGroup
Adult        864
MiddleAge    243
Child        102
Teen          60
Senior        40
Name: count, dtype: int64

In [26]:
skew_before = clean["Fare"].skew()
clean["FareLog"] = np.log1p(clean["Fare"])
skew_after = clean["FareLog"].skew()

print(f"Fare skewness before transform: {skew_before:.2f}")
print(f"Fare skewness after transform : {skew_after:.2f}")

Fare skewness before transform: 4.37
Fare skewness after transform : 0.54


In [27]:
category_cols = ["Sex", "Embarked", "Title", "Deck", "AgeGroup"]
for col in category_cols:
    clean[col] = clean[col].astype("category")


column_order = [
    "PassengerId", "Survived",
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
    "Name", "Ticket",
    "Title", "AgeGroup", "AgeWasMissing", "FamilySize", "IsAlone",
    "HasCabin", "Deck", "FareLog", "FareWasZero",
]
clean = clean[column_order]

clean.dtypes

PassengerId         int64
Survived            Int64
Pclass              int64
Sex              category
Age               float64
SibSp               int64
Parch               int64
Fare              float64
Embarked         category
Name               object
Ticket             object
Title            category
AgeGroup         category
AgeWasMissing        bool
FamilySize          int64
IsAlone             int64
HasCabin            int64
Deck             category
FareLog           float64
FareWasZero          bool
dtype: object

### 2c. Validate and save

In [28]:
still_missing = clean.isna().sum()
still_missing = still_missing[still_missing > 0]
print("Columns that still contain missing values:")
print(still_missing)

assert set(still_missing.index) == {"Survived"}, "Unexpected missing values remain"
assert clean["PassengerId"].is_unique, "PassengerId is not unique"
assert len(clean) == len(raw), "Row count changed during cleaning"
assert (clean["Age"] > 0).all(), "Non-positive age found"
assert (clean["Fare"] >= 0).all(), "Negative fare found"
assert clean["FamilySize"].min() >= 1, "Family size below 1"
print("\nAll validation checks passed.")

Columns that still contain missing values:
Survived    418
dtype: int64

All validation checks passed.


In [29]:

before = raw.isna().sum()
after = clean.isna().sum().reindex(before.index).astype("Int64")
pd.DataFrame({"missing_before": before, "missing_after": after})

,missing_before,missing_after
PassengerId,0,0
Survived,418,418
Pclass,0,0
Name,0,0
Sex,0,0
Age,263,0
SibSp,0,0
Parch,0,0
Ticket,0,0
Fare,1,0


In [30]:
raw_mb = raw.memory_usage(deep=True).sum() / 1024**2
clean_mb = clean.memory_usage(deep=True).sum() / 1024**2
print(f"Raw data    : {raw_mb:.2f} MB across {raw.shape[1]} columns")
print(f"Cleaned data: {clean_mb:.2f} MB across {clean.shape[1]} columns")

Raw data    : 0.41 MB across 12 columns
Cleaned data: 0.29 MB across 20 columns


In [31]:
labeled = clean[clean["Survived"].notna()].copy()
holdout = clean[clean["Survived"].isna()].copy()
print(f"Labeled rows (analysis set): {len(labeled)}")
print(f"Holdout rows               : {len(holdout)}")

pd.crosstab(labeled["Sex"], labeled["Pclass"], margins=True)

Labeled rows (analysis set): 891
Holdout rows               : 418


Pclass,1,2,3,All
Sex,,,,
female,94,76,144,314
male,122,108,347,577
All,216,184,491,891


In [32]:
clean.to_csv("titanic_clean.csv", index=False)
print("Saved titanic_clean.csv with shape", clean.shape)

clean.head()

Saved titanic_clean.csv with shape (1309, 20)


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Name,Ticket,Title,AgeGroup,AgeWasMissing,FamilySize,IsAlone,HasCabin,Deck,FareLog,FareWasZero
0,1,0,3,male,22.0,1,0,7.2500,S,"Braund, Mr. Owen Harris",A/5 21171,Mr,Adult,False,2,0,0,U,2.110213,False
1,2,1,1,female,38.0,1,0,71.2833,C,"Cumings, Mrs. John Bradley (Florence Briggs Th...",PC 17599,Mrs,Adult,False,2,0,1,C,4.280593,False
2,3,1,3,female,26.0,0,0,7.9250,S,"Heikkinen, Miss. Laina",STON/O2. 3101282,Miss,Adult,False,1,1,0,U,2.188856,False
3,4,1,1,female,35.0,1,0,53.1000,S,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",113803,Mrs,Adult,False,2,0,1,C,3.990834,False
4,5,0,3,male,35.0,0,0,8.0500,S,"Allen, Mr. William Henry",373450,Mr,Adult,False,1,1,0,U,2.202765,False


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Name,Ticket,Title,AgeGroup,AgeWasMissing,FamilySize,IsAlone,HasCabin,Deck,FareLog,FareWasZero
0,1,0,3,male,22.0,1,0,7.2500,S,"Braund, Mr. Owen Harris",A/5 21171,Mr,Adult,False,2,0,0,U,2.110213,False
1,2,1,1,female,38.0,1,0,71.2833,C,"Cumings, Mrs. John Bradley (Florence Briggs Th...",PC 17599,Mrs,Adult,False,2,0,1,C,4.280593,False
2,3,1,3,female,26.0,0,0,7.9250,S,"Heikkinen, Miss. Laina",STON/O2. 3101282,Miss,Adult,False,1,1,0,U,2.188856,False
3,4,1,1,female,35.0,1,0,53.1000,S,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",113803,Mrs,Adult,False,2,0,1,C,3.990834,False
4,5,0,3,male,35.0,0,0,8.0500,S,"Allen, Mr. William Henry",373450,Mr,Adult,False,1,1,0,U,2.202765,False


---
# Summary of the Prepare Step

| Issue found | Rows affected | Action taken | Reason |
|---|---|---|---|
| `Survived` blank | 418 | Kept, converted to nullable `Int64`, split into labeled and holdout sets | Blank outcomes cannot test the hypothesis, but the rows still help imputation |
| `Age` missing | 263 | Filled with median age of the same class and title, with an `AgeWasMissing` flag | Gaps were uneven across classes, so one global median would bias results |
| `Cabin` missing | 1,014 | Replaced with `HasCabin` and `Deck`, raw column dropped | Too sparse to fill without inventing data |
| `Embarked` missing | 2 | Filled with the most common port (S) | Both passengers shared a ticket, and S is the clear majority |
| `Fare` missing | 1 | Filled with the median fare of the passenger's class | Fare depends strongly on class |
| `Fare` equal to 0 | 17 | Kept and flagged in `FareWasZero` | Plausibly unrecorded or complimentary, so not safe to overwrite |
| `Fare` skewed | all | Added `FareLog` using `np.log1p` | Reduces the pull of the long right tail of expensive tickets |
| Repeated names | 4 | Kept | Different tickets and ages show they are different people |
| New features | all | `Title`, `AgeGroup`, `FamilySize`, `IsAlone` | Give later steps meaningful ways to describe passengers |

**Result:** The cleaned dataset has the same 1,309 passengers, no missing values other than the known holdout outcomes, consistent data types, and documented decisions for every change. It is saved as `titanic_clean.csv`.
